# haRdQualizer — Design ONE realistic goth character ("her")

Step 1 of the realistic-character plan: define and lock a single **original**
goth woman with a distinctive, reproducible look, then export a clean full-body
`body.png` for the visualizer. The few extra shots this notebook makes also seed
the dataset for a future character LoRA (step 2), which is what gives true
consistency across many poses/animations.

Uses **RealVisXL V5.0** (a strong photoreal SDXL model) on a Colab **T4**.

### Scope / ethics
An **original, fictional** character in a goth aesthetic. Do **not** reproduce a
real person's likeness — references are a style moodboard only.

### Flow
1. *Casting* (cell 6): generate several candidates; pick the SEED you like.
2. *Lock her* (cell 7): set `CHOSEN_SEED`, render the hero shot + a few angles
   with that seed to check she stays recognizable.
3. *Export* (cells 8-9): cut out the background and download her PNGs.

In [ ]:
# 1. Check the GPU (expect a Tesla T4, ~15GB).
!nvidia-smi

In [ ]:
# 2. Install dependencies. Pin pillow<12 so rembg doesn't break diffusers.
!pip install -q diffusers transformers accelerate safetensors rembg onnxruntime "pillow<12"
print("\n*** If pip changed Pillow: Runtime > Restart session, then run from cell 3.")

In [ ]:
# 3. Guard: a diffusers-compatible Pillow must be active.
import PIL
assert int(PIL.__version__.split('.')[0]) < 12, (
    'Pillow >= 12 loaded -> Runtime > Restart session, then run from here.')
print('Pillow', PIL.__version__, 'OK')

In [ ]:
# 4. Load RealVisXL V5.0 on the GPU (DPM++ 2M Karras sampler).
import gc
import torch
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

MODEL_ID = "SG161222/RealVisXL_V5.0"

pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, variant="fp16",
    use_safetensors=True, low_cpu_mem_usage=True,
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config, use_karras_sigmas=True, algorithm_type="dpmsolver++")
pipe = pipe.to("cuda")
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()
gc.collect(); torch.cuda.empty_cache()
print("ready on", pipe.device, f"| VRAM {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# 5. Define HER. Distinctive, reproducible identity = stays recognizable.
#    Edit freely, but keep concrete, unusual features (eye colour, beauty mark,
#    hair) — they anchor her identity across seeds and later in the LoRA.
CHAR_NAME = "goth_d"

APPEARANCE = (
    "a 22 year old gothic woman, oval face, very pale porcelain skin, large dark "
    "grey almond eyes, sharp winged black eyeliner, thin arched eyebrows, small "
    "straight nose, full dark plum lips, a small beauty mark below the left eye, "
    "long straight jet-black hair with blunt bangs"
)
OUTFIT = (
    "wearing a black gothic outfit: spiked leather choker, sheer black long-sleeve "
    "top, black corset, black pleated skirt, fishnet tights, chunky platform boots, "
    "silver chain jewelry"
)
STYLE = (
    "RAW full body photograph, standing, facing camera, full figure head to feet, "
    "soft studio lighting, plain light grey background, sharp focus, detailed skin "
    "texture, shot on 85mm lens, photorealistic, high detail"
)
NEG = (
    "anime, cartoon, illustration, drawing, 3d render, cgi, doll, plastic skin, "
    "deformed, disfigured, bad anatomy, bad hands, extra fingers, fused fingers, "
    "missing limbs, extra limbs, cropped, head out of frame, blurry, lowres, "
    "worst quality, low quality, jpeg artifacts, watermark, text, signature, "
    "nsfw, nude, multiple people, 2girls, child"
)

HERO = APPEARANCE + ", " + OUTFIT + ", arms relaxed at her sides, " + STYLE

CFG = 5.0
STEPS = 30
W, H = 832, 1216

def generate(prompt, seed, w=W, h=H, steps=STEPS):
    g = torch.Generator(device="cuda").manual_seed(int(seed))
    img = pipe(prompt=prompt, negative_prompt=NEG, num_inference_steps=steps,
               guidance_scale=CFG, width=w, height=h, generator=g).images[0]
    torch.cuda.empty_cache()
    return img
print("character defined:", CHAR_NAME)

In [ ]:
# 6. CASTING — generate candidates so you can pick the SEED you like best.
#    (Smaller/faster here; the winner is re-rendered full-size in cell 7.)
import matplotlib.pyplot as plt

CAST_SEEDS = [1, 7, 42, 101, 777, 2024]
cast = {s: generate(HERO, s, w=640, h=960, steps=26) for s in CAST_SEEDS}

cols = len(cast)
fig, axes = plt.subplots(1, cols, figsize=(3 * cols, 6))
for ax, (s, im) in zip(axes, cast.items()):
    ax.imshow(im); ax.set_title(f"seed {s}"); ax.axis("off")
plt.tight_layout(); plt.show()
print("Pick the seed you like, set CHOSEN_SEED in the next cell.")

In [ ]:
# 7. LOCK HER — set the winning seed; render the hero + a few angles with it.
#    Same seed + same appearance keeps her roughly consistent (the LoRA in
#    step 2 makes it exact). These shots also seed that future dataset.
from pathlib import Path

CHOSEN_SEED = 42   # <-- set to your favourite from the casting grid

SHOTS = {
    "body":   "arms relaxed at her sides, facing camera, neutral expression",
    "threeq": "three quarter view, hand on hip, slight confident smile",
    "profile": "side profile view, looking to the side",
    "armsup": "both arms raised overhead, dynamic pose",
}
RAW = Path("/content/her_raw"); RAW.mkdir(exist_ok=True)
raw_shots = {}
for tag, pose in SHOTS.items():
    prompt = APPEARANCE + ", " + OUTFIT + ", " + pose + ", " + STYLE
    img = generate(prompt, CHOSEN_SEED)
    img.save(RAW / f"{tag}.png"); raw_shots[tag] = img
    print("rendered", tag)

fig, axes = plt.subplots(1, len(raw_shots), figsize=(3 * len(raw_shots), 6))
for ax, (tag, im) in zip(axes, raw_shots.items()):
    ax.imshow(im); ax.set_title(tag); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# 8. Cut out the background (human segmentation) -> transparent PNGs.
#    u2net_human_seg is tuned for real people (better than the anime matte here).
from rembg import remove, new_session

human = new_session("u2net_human_seg")
OUT = Path("/content/her") / CHAR_NAME
OUT.mkdir(parents=True, exist_ok=True)
DATASET = Path("/content/her_dataset"); DATASET.mkdir(exist_ok=True)

for tag, img in raw_shots.items():
    cut = remove(img, session=human)
    # The hero shot becomes body.png for the rig; the rest seed the LoRA set.
    if tag == "body":
        cut.save(OUT / "body.png"); print("saved", OUT / "body.png")
    img.save(DATASET / f"{CHAR_NAME}_{tag}.png")  # keep raw (with bg) for LoRA
print("hero + dataset shots ready")

In [ ]:
# 9. Package and download.
#    her/<CHAR_NAME>/body.png  -> drop into haRdQualizer/assets/characters/
#    her_dataset/*.png         -> keep for the future character-LoRA (step 2)
import shutil
from google.colab import files

shutil.make_archive("/content/her", "zip", "/content/her")
shutil.make_archive("/content/her_dataset", "zip", "/content/her_dataset")
print("Extract her.zip into assets/characters/ ; keep her_dataset.zip for the LoRA.")
files.download("/content/her.zip")
files.download("/content/her_dataset.zip")